# Week 5-2 · MMT-03 — Overview of Electronic & Algorithmic Trading
**Instructor: Rob Cassell.** Session 1 of 2 — the *concepts* of the algorithmic trading process.
(The math models come in MMT-04.) This notebook turns every idea into a runnable simulation that
reproduces the **exact numbers** from the lecture.

**The story.** An institution must trade a **100,000-share** order. Dump it on the market at once and you
*move the price against yourself* (market impact). Trade it too slowly and *the market moves against you*
(timing risk). Everything in this lecture is about balancing those two forces — the **trader's dilemma** —
and about *where* and *how* the sliced child-orders get executed.

**What we reproduce:**
1. **Market impact** decomposed into temporary + permanent (the $30 → $30.25 → $30.05 path)
2. The **trader's dilemma** — minimise `MI(t) + λ·TR(t)` → optimal trade horizon ≈ 0.85 day
3. **Slicing schedules**: VWAP (J-curve), TWAP (uniform), POV (% of volume)
4. The **trade-cost distribution** & the agency-vs-principal-bid 70% decision
5. The **order book**, **NBBO** across 3 venues, and the **dark-pool midpoint** = 30.04
6. **Smart order routing** — why the longest queue can be the fastest fill
7. **Maker–taker** venue economics in mills


In [1]:
import numpy as np
import pandas as pd
import math
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
np.random.seed(3)
def norm_cdf(z):                 # standard-normal CDF without scipy
    return 0.5*(1+math.erf(z/math.sqrt(2)))
print("ready")


ready


## Part 1 — Market impact = temporary + permanent

A large **buy** pushes the price up while you trade; when you stop, the price *reverts* — but not all the
way. The leftover is **permanent impact** (the market infers information from your order); the part that
snaps back is **temporary impact** (the liquidity premium you paid to attract sellers).

Lecture example: stock at **\$30.00**, your buying lifts it to **\$30.25** (total impact 25¢), then it
settles at a new equilibrium of **\$30.05**.

In [2]:
p0, peak, equil = 30.00, 30.25, 30.05
total_impact = peak - p0
permanent     = equil - p0
temporary     = peak - equil
print(f"total impact   = {total_impact*100:.0f} cents")
print(f"  temporary    = {temporary*100:.0f} cents  (liquidity premium, reverts)")
print(f"  permanent    = {permanent*100:.0f} cents  (information content, stays)")


total impact   = 25 cents
  temporary    = 20 cents  (liquidity premium, reverts)
  permanent    = 5 cents  (information content, stays)


### Bridge note 1

The next code cell continues the same worked example. This markdown cell is added only to make the validated notebook self-explaining before each runnable block.

In [3]:
# illustrate the price path during/after the trade
t = np.arange(0, 60)
price = np.where(t < 30, p0 + total_impact*(t/29),          # ramp up while trading
                 equil + temporary*np.exp(-(t-30)/6))        # revert toward equilibrium
fig, ax = plt.subplots(figsize=(8,3.4))
ax.plot(t, price, color="#0891b2", lw=2)
ax.axhline(p0,    ls=":", color="#64748b"); ax.axhline(equil, ls="--", color="#16a34a")
ax.annotate("peak 30.25", (29,30.25), color="#0891b2")
ax.annotate("new equilibrium 30.05", (40,30.05), color="#16a34a")
ax.annotate("temporary\n(20c)", (32,30.15), color="#b45309")
ax.set_title("Buy order — market impact then reversion"); ax.set_xlabel("time"); ax.set_ylabel("price $")
plt.tight_layout(); plt.savefig("chart_1_impact.png", dpi=110); plt.show(); print("saved chart_1_impact.png")


saved chart_1_impact.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_25944\1594907567.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_1_impact.png", dpi=110); plt.show(); print("saved chart_1_impact.png")


## Part 2 — The trader's dilemma

**Market impact falls** the slower you trade (you leak the order out gently). **Timing risk rises** the
slower you trade (more hours for the price to drift against you). Your cost is

$$\text{Cost}(t) = \underbrace{MI(t)}_{\downarrow\ \text{with } t} + \lambda\,\underbrace{TR(t)}_{\uparrow\ \text{with } t}$$

where **λ is your risk aversion**: `λ=1` weighs impact and risk equally, `λ>1` = risk-averse (trade
faster), `λ<1` = risk-neutral (trade slower). We pick simple functional forms calibrated so the
equal-weight solution lands at the lecture's **≈0.85 trading days**.

In [4]:
t = np.linspace(0.05, 4.0, 600)                 # trade horizon in days
MI = 0.17 / np.sqrt(t)                           # impact decreasing with time (sqrt law)
TR = 0.20 * np.sqrt(t)                            # risk increasing with sqrt(time)
# closed form: minimise a/sqrt(t)+lam*b*sqrt(t) -> t* = a/(lam*b); with a=0.17,b=0.20, lam=1 -> 0.85

def optimal_t(lam):
    cost = MI + lam*TR
    return t[np.argmin(cost)], cost
for lam in [0.25, 1.0, 3.0]:
    t_star,_ = optimal_t(lam)
    tag = "risk-neutral" if lam<1 else ("risk-averse" if lam>1 else "balanced")
    print(f"lambda={lam:>4}  ({tag:<12}) -> optimal horizon = {t_star:.2f} days")


lambda=0.25  (risk-neutral) -> optimal horizon = 3.40 days
lambda= 1.0  (balanced    ) -> optimal horizon = 0.85 days
lambda= 3.0  (risk-averse ) -> optimal horizon = 0.28 days


### Bridge note 2

The next code cell continues the same worked example. This markdown cell is added only to make the validated notebook self-explaining before each runnable block.

In [5]:
fig, ax = plt.subplots(figsize=(8,3.6))
ax.plot(t, MI, color="#0891b2", label="market impact (↓ with time)")
ax.plot(t, TR, color="#ea580c", label="timing risk (↑ with time)")
cost = MI + 1.0*TR
ax.plot(t, cost, color="#7c3aed", lw=2, label="total cost (λ=1)")
ts,_ = optimal_t(1.0)
ax.axvline(ts, ls="--", color="#16a34a"); ax.annotate(f"optimal ≈ {ts:.2f} d", (ts+.03, cost.max()*0.7), color="#16a34a")
ax.set_title("Trader's dilemma — balancing impact vs risk"); ax.set_xlabel("trade horizon (days)"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_2_dilemma.png", dpi=110); plt.show(); print("saved chart_2_dilemma.png")


saved chart_2_dilemma.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_25944\3086680749.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_2_dilemma.png", dpi=110); plt.show(); print("saved chart_2_dilemma.png")


## Part 3 — Slicing schedules: VWAP, TWAP, POV

Once you know *how long* to trade, you decide *how many shares each interval*. Three classic schedules
slice the 100,000-share order across the trading day:
- **TWAP** — time-weighted: equal shares each bucket (a uniform distribution).
- **VWAP** — volume-weighted: follow the day's intraday volume profile (a **J-curve** — heavy at the open,
  heaviest at the close).
- **POV** — percentage of volume: trade a fixed % (e.g. 10%) of whatever volume prints.

In [6]:
order = 100_000
buckets = pd.date_range("09:30", "16:00", freq="30min")
n = len(buckets)
# stylised intraday volume profile (J-shape: high open, dip midday, highest close)
x = np.linspace(0, 1, n)
vol_profile = 0.6 + 1.4*np.exp(-((x-0.0)/0.18)**2) + 2.2*np.exp(-((x-1.0)/0.14)**2) + 0.4
vol_profile /= vol_profile.sum()

twap = np.full(n, order/n)
vwap = order * vol_profile
sched = pd.DataFrame({"bucket":[b.strftime("%H:%M") for b in buckets],
                      "mkt_vol_%":(vol_profile*100).round(1),
                      "TWAP":twap.round(0), "VWAP":vwap.round(0)})
print(sched.to_string(index=False))
print(f"\ncheck: TWAP sum={twap.sum():.0f}  VWAP sum={vwap.sum():.0f}  (target {order})")


bucket  mkt_vol_%   TWAP    VWAP
 09:30       10.8 7143.0 10786.0
 10:00        9.7 7143.0  9735.0
 10:30        7.5 7143.0  7524.0
 11:00        5.7 7143.0  5710.0
 11:30        4.8 7143.0  4833.0
 12:00        4.6 7143.0  4559.0
 12:30        4.5 7143.0  4503.0
 13:00        4.5 7143.0  4495.0
 13:30        4.5 7143.0  4499.0
 14:00        4.6 7143.0  4573.0
 14:30        5.1 7143.0  5147.0
 15:00        7.4 7143.0  7449.0
 15:30       11.8 7143.0 11805.0
 16:00       14.4 7143.0 14381.0

check: TWAP sum=100000  VWAP sum=100000  (target 100000)


### Bridge note 3

The next code cell continues the same worked example. This markdown cell is added only to make the validated notebook self-explaining before each runnable block.

In [7]:
pov_rate = 0.10
day_volume = order / pov_rate                       # if we are 10% of volume, total mkt vol = 1,000,000
pov_shares = vol_profile * day_volume * pov_rate
print(f"POV 10%: total market volume implied = {day_volume:,.0f} -> our shares = {pov_shares.sum():,.0f}")

fig, ax = plt.subplots(figsize=(8,3.4))
w=0.4; idx=np.arange(n)
ax.bar(idx-w/2, twap, width=w, color="#94a3b8", label="TWAP (uniform)")
ax.bar(idx+w/2, vwap, width=w, color="#0891b2", label="VWAP (J-curve)")
ax.set_xticks(idx); ax.set_xticklabels([b.strftime("%H:%M") for b in buckets], rotation=45, fontsize=7)
ax.set_title("Slicing the 100,000-share order"); ax.set_ylabel("shares"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig("chart_3_schedules.png", dpi=110); plt.show(); print("saved chart_3_schedules.png")


POV 10%: total market volume implied = 1,000,000 -> our shares = 100,000


saved chart_3_schedules.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_25944\1120599956.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_3_schedules.png", dpi=110); plt.show(); print("saved chart_3_schedules.png")


## Part 4 — Trade-cost distribution: agency vs principal bid

An **agency** execution costs the *expected* price but carries risk; a **principal bid** is a guaranteed
price for a fat premium. The lecture's numbers:

- current price \$30.00, market impact **10¢**, commission **2¢** → expected all-in **\$30.12**
- timing risk (std-dev) **±20¢**
- a principal bid charges **25¢** commission → all-in **\$30.25**

The question: *what's the probability the agency execution beats the 25¢ bid?* Read it off the normal CDF.

In [8]:
mean_cost = 30.00 + 0.10 + 0.02     # 30.12 expected agency all-in
sigma     = 0.20                     # timing risk
principal = 30.00 + 0.25             # 30.25 guaranteed

z = (principal - mean_cost) / sigma
p_agency_better = norm_cdf(z)
print(f"agency expected all-in = ${mean_cost:.2f}")
print(f"principal bid all-in   = ${principal:.2f}")
print(f"z = (30.25 - 30.12)/0.20 = {z:.2f}")
print(f"P(agency cost <= principal bid) = {p_agency_better:.1%}   -> ~70% as in lecture")


agency expected all-in = $30.12
principal bid all-in   = $30.25
z = (30.25 - 30.12)/0.20 = 0.65
P(agency cost <= principal bid) = 74.2%   -> ~70% as in lecture


### Bridge note 4

The next code cell continues the same worked example. This markdown cell is added only to make the validated notebook self-explaining before each runnable block.

In [9]:
grid = np.linspace(mean_cost-3*sigma, mean_cost+3*sigma, 400)
pdf = np.exp(-0.5*((grid-mean_cost)/sigma)**2)/(sigma*np.sqrt(2*np.pi))
fig, (a1,a2) = plt.subplots(1,2,figsize=(9,3.3))
a1.plot(grid, pdf, color="#0891b2"); a1.axvline(mean_cost, ls="--", color="#7c3aed", label="mean 30.12")
a1.axvline(principal, ls="--", color="#ea580c", label="bid 30.25")
a1.fill_between(grid, pdf, where=grid<=principal, color="#a5f3fc", alpha=.6)
a1.set_title("Trade-cost distribution"); a1.legend(fontsize=7)
cdf = np.array([norm_cdf((g-mean_cost)/sigma) for g in grid])
a2.plot(grid, cdf, color="#0891b2"); a2.axvline(principal, ls="--", color="#ea580c")
a2.axhline(p_agency_better, ls=":", color="#16a34a")
a2.annotate(f"{p_agency_better:.0%}", (principal+0.01, p_agency_better-0.08), color="#16a34a")
a2.set_title("Cumulative — P(cost ≤ x)")
plt.tight_layout(); plt.savefig("chart_4_costdist.png", dpi=110); plt.show(); print("saved chart_4_costdist.png")


saved chart_4_costdist.png


C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_25944\1216554118.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("chart_4_costdist.png", dpi=110); plt.show(); print("saved chart_4_costdist.png")


**The subtlety the lecture stresses:** 70% sounds great, but it depends on *how often you trade*. Trade
daily and a 70% edge compounds in your favour. Trade *once a year* and a 30% chance of a worse fill — with
a 12-month wait to recover — may not be worth it. Frequency, not just probability, drives the choice.

## Part 5 — Order book, NBBO, and the dark-pool midpoint

Each venue keeps an **order book**: bids (buyers) sorted high→low, asks (sellers) low→high, executed by
**price–time priority**. Across venues the **NBBO** (national best bid & offer) takes the highest bid and
lowest ask. **Dark pools** print at the **NBBO midpoint** — fair by construction, but their book is hidden.

In [10]:
venues = {
  "A": {"bid":30.00, "bid_sz":800,  "ask":30.10, "ask_sz":300},
  "B": {"bid":30.02, "bid_sz":200,  "ask":30.15, "ask_sz":600},
  "C": {"bid":29.90, "bid_sz":1600, "ask":30.06, "ask_sz":100},
}
book = pd.DataFrame(venues).T
print(book.to_string())
best_bid = book["bid"].max();  best_bid_v = book["bid"].idxmax()
best_ask = book["ask"].min();  best_ask_v = book["ask"].idxmin()
mid = (best_bid + best_ask)/2
print(f"\nNBBO best bid = {best_bid:.2f} (venue {best_bid_v}) | best ask = {best_ask:.2f} (venue {best_ask_v})")
print(f"dark-pool execution price = NBBO midpoint = {mid:.2f}")


     bid  bid_sz    ask  ask_sz
A  30.00   800.0  30.10   300.0
B  30.02   200.0  30.15   600.0
C  29.90  1600.0  30.06   100.0

NBBO best bid = 30.02 (venue B) | best ask = 30.06 (venue C)
dark-pool execution price = NBBO midpoint = 30.04


## Part 6 — Smart order routing: the longest queue can be fastest

You want to **buy 100 shares at \$30**. Under price–time priority you must wait behind everyone already
queued at that price on a venue. Naïvely you pick the **shortest queue** — but the right answer also
weighs **how fast each venue trades**. Venue A has the longest queue yet trades **10×** the volume.

In [11]:
# shares ahead of you at $30 on each venue, and each venue's relative trade speed
queue_ahead = {"A":2600, "B":800, "C":1400}
speed       = {"A":10.0, "B":1.0, "C":1.0}      # A trades 10x faster
wait = {v: queue_ahead[v]/speed[v] for v in queue_ahead}
tbl = pd.DataFrame({"shares_ahead":queue_ahead, "rel_speed":speed,
                    "expected_wait":pd.Series(wait).round(0)})
print(tbl.to_string())
best = min(wait, key=wait.get)
print(f"\nshortest *queue*  -> venue {min(queue_ahead, key=queue_ahead.get)}")
print(f"shortest *wait*   -> venue {best}  (queue/speed wins)")


   shares_ahead  rel_speed  expected_wait
A          2600       10.0          260.0
B           800        1.0          800.0
C          1400        1.0         1400.0

shortest *queue*  -> venue B
shortest *wait*   -> venue A  (queue/speed wins)


## Part 7 — How venues get paid: maker–taker economics

Fees are quoted in **mills** (1 mill = 1/100 of a penny = \$0.0001). Three models:
- **Commission** (common in dark pools): *both* sides pay a fee.
- **Maker–taker**: the **maker** (posts a limit order, adds liquidity) gets a **rebate**; the **taker**
  (market order, removes liquidity) pays a fee. The venue keeps the spread.
- **Inverted maker–taker**: reversed — the taker gets the rebate (a "pay to cut the line" model).

In [12]:
# (who pays the fee, who gets the rebate, fee mills, rebate mills)
models = [
  ("commission (dark pool)", "both sides", "-",     10, 0),   # buyer 10 + seller 10
  ("maker-taker",            "taker",      "maker", 30, 10),  # taker pays 30, maker rebate 10
  ("inverted maker-taker",   "maker",      "taker", 30, 10),  # maker pays 30, taker rebate 10
]
for name, payer, rebated, fee, rebate in models:
    if name.startswith("commission"):
        print(f"{name:<24}: both sides pay {fee} mills -> venue keeps {2*fee} mills")
    else:
        print(f"{name:<24}: {payer} pays {fee} mills, {rebated} gets {rebate} rebate -> venue keeps {fee-rebate} mills")


commission (dark pool)  : both sides pay 10 mills -> venue keeps 20 mills
maker-taker             : taker pays 30 mills, maker gets 10 rebate -> venue keeps 20 mills
inverted maker-taker    : maker pays 30 mills, taker gets 10 rebate -> venue keeps 20 mills


### Bridge note 5

The next code cell continues the same worked example. This markdown cell is added only to make the validated notebook self-explaining before each runnable block.

In [13]:
# put a dollar value on it for our 100,000-share order at maker-taker (we post limit = maker)
shares = 100_000
rebate_per_share = 10 * 1e-4          # 10 mills
print(f"if we MAKE liquidity on 100,000 shares: rebate = {shares*rebate_per_share:,.2f} $  (10 mills/share)")
print(f"if we TAKE liquidity instead:            fee    = {shares*30*1e-4:,.2f} $  (30 mills/share)")


if we MAKE liquidity on 100,000 shares: rebate = 100.00 $  (10 mills/share)
if we TAKE liquidity instead:            fee    = 300.00 $  (30 mills/share)


## Summary — the algorithmic trading process
- **Market impact** = temporary (liquidity premium, reverts) + permanent (information, stays): 25¢ = 20¢ + 5¢.
- **Trader's dilemma:** minimise `MI(t) + λ·TR(t)`; impact ↓ with time, risk ↑ with time; optimal ≈ 0.85 day; λ tunes urgency.
- **Slicing schedules:** TWAP (uniform), VWAP (follow J-curve volume), POV (fixed % of volume).
- **Agency vs principal bid:** agency expected \$30.12 ±20¢ vs guaranteed \$30.25 → ~70% chance agency wins — but *trade frequency* decides.
- **Venues:** exchanges & ATSs are displayed; **dark pools** are hidden and print at the **NBBO midpoint (30.04)**.
- **Smart order routing:** balance queue depth *and* venue speed — the longest queue (A) can fill fastest (10× speed).
- **Maker–taker:** makers (limit/add liquidity) earn rebates; takers (market/remove) pay fees; mills = \$0.0001.

> Session 2 (MMT-04) derives the **market-impact and timing-risk models** themselves and shows how to
> measure broker/algo performance — in Python.


---

# Additive resource coverage

The following cells append the official source gaps from the lecture notes and summary without changing the original worked examples.

## Setup

Load the reference tables created from the lecture summary and lecture note.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
print("setup complete")

setup complete


## Financial market participants

The execution decision is made inside a broader ecosystem: institutions generate large orders, brokers facilitate execution, venues host liquidity, and liquidity/HFT firms often supply or remove displayed liquidity.

In [15]:
participants = pd.read_csv("mmt03_market_participants.csv")
display(participants)

,participant,primary_role,typical_motivation,examples
0,Brokers,facilitate client execution,"commissions, principal bid premium, client ser...","investment banks, full-service, bulge bracket,..."
1,Institutions,invest and rebalance portfolios,implementation consistent with fund objective,"asset managers, mutual funds, index funds, ins..."
2,Hedge funds,seek alpha and manage risk,quant and non-quant investment profits,"long/short, statistical arbitrage, event strat..."
3,Liquidity providers,post bids and asks,spread capture and venue rebates,"specialists, market makers, NYSE DMM, Nasdaq Q..."
4,Retail investors,individual trading and investing,"portfolio allocation, wealth management, specu...","individuals, wealth managers, Gen-X, millennia..."
5,Proprietary trading firms,trade firm capital,short-horizon trading profit,"prop firms, market-making firms"
6,High-frequency traders,fast liquidity and pattern trading,"spread, rebates, short-term order-flow patterns",HFT liquidity and rebate strategies


## Algorithm families

MMT-03 distinguishes execution-only algorithms from profit-seeking algorithms, market-making/HFT logic, liquidity-seeking logic, and direct market access.

In [16]:
taxonomy = pd.read_csv("mmt03_algorithm_taxonomy.csv")
display(taxonomy)

,family,decision_scope,typical_user,key_controls
0,Execution-only,execute an order already decided by the portfo...,institutional buy side,"schedule, urgency, limit/market mix, venue rou..."
1,Quant / profit-seeking,decide the investment and execute it,"quant funds, prop firms","signal model, risk limits, immediate execution..."
2,Auto market making / HFT,post bids and asks to provide liquidity,"market makers, HFT firms","spread, inventory, rebates, adverse selection"
3,Liquidity trading,find and capture available liquidity,HFT and broker liquidity algos,"venue mix, displayed/dark split, pattern detec..."
4,Direct market access,client uses broker connectivity with own logic,sophisticated institutions,"connectivity, risk checks, venue access"


## Urgency, static updates, and dynamic updates

Urgency describes how aggressively the algorithm should trade; updating describes whether the algorithm can revise itself without a manual trader change.

In [17]:
strategy = pd.read_csv("mmt03_strategy_classification.csv")
display(strategy)

,classification,definition,order_mix,examples
0,Aggressive,get me done; sweep all liquidity at my price o...,more market orders,"urgent sweep, completion trade"
1,Working order,balance market impact and timing risk,market and limit orders,"VWAP, TWAP, POV, IS, optimal strategy"
2,Passive,"use crossing systems and dark pools, wait for ...",more limit orders,"liquidity seeking, dark-pool posting"
3,Static,parameters do not change unless the trader rev...,fixed by initial setup,"fixed TWAP, fixed POV"
4,Dynamic,algorithm revises rate/parameters from live ma...,"changes with prices, volume, and optimization","dynamic POV, arrival price, re-optimized schedule"


## Intraday microstructure profiles

The transcript emphasizes that intraday volume is now more J-shaped, volatility is closer to L-shaped, and spreads are widest near the open/close. Execution algorithms use these shapes when deciding schedule, limit prices, and routing urgency.

In [18]:
profiles = pd.read_csv("mmt03_intraday_profiles.csv")
fig, ax1 = plt.subplots(figsize=(10, 4.5))
ax1.bar(profiles["bucket"], profiles["volume_pct"], color="#2563eb", alpha=0.8, label="Volume %")
ax1.set_ylabel("Volume share of day (%)")
ax1.tick_params(axis="x", rotation=45)
ax2 = ax1.twinx()
ax2.plot(profiles["bucket"], profiles["relative_volatility"], color="#ef4444", marker="o", label="Relative volatility")
ax2.plot(profiles["bucket"], profiles["relative_spread"], color="#f59e0b", marker="s", label="Relative spread")
ax2.set_ylabel("Relative level")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.title("Intraday volume, volatility, and spread profiles")
plt.tight_layout()
plt.show()

C:\Users\hsaeed\AppData\Local\Temp\1\ipykernel_25944\3438340847.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Broker execution style

Agency and principal bid decisions are not only a one-trade expected value problem. The lecture stresses frequency: a 70% chance of agency beating the principal bid is very different for daily trading than for one annual trade.

In [19]:
styles = pd.read_csv("mmt03_execution_styles.csv")
display(styles)

,style,who_bears_risk,broker_compensation,price_certainty
0,Agency execution,investor,"commission, often 1-3 cents/share",unknown until execution completes
1,Principal bid / capital commitment,broker after committing price,"premium, often much higher such as 5-25 cents/...",guaranteed benchmark or all-in price


## Algorithmic decision process

The implementation process links the macro schedule to the micro order-placement problem.

In [20]:
decision = pd.read_csv("mmt03_decision_process.csv")
display(decision)

,step,process_step,question
0,1,Select broker,"Who can execute with the required quality, ven..."
1,2,Select algorithm and parameters,"What schedule, risk aversion, urgency, and ben..."
2,3,Transact using LOM and SOR,What market/limit mix and venue allocation bes...
3,4,Revise if necessary,"Do live prices, volumes, fills, or risk requir..."
4,5,Measure cost and evaluate performance,Did the broker/algorithm deliver quality relat...
